In [1]:
import pandas as pd
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote_plus
from rdflib.namespace import RDF, DC, Namespace
import xml.etree.ElementTree as ET
from lxml import etree
import shutil
import zipfile
import ftplib
import io
import csv
import field_extractor as fe

In [2]:
# Constants
UNZIP_DIR = "selected_data"
FTP_HOST = "download.europeana.eu"
FTP_PATH = "dataset/XML/"
OUTPUT_DIR = "collected_data"

In [3]:
data_ids = []
# extract the dataset_ids
with open('dataset_ids.txt', 'r') as f:
    data_ids = f.readlines()
    data_ids = [x.strip() for x in data_ids]

data_ids = data_ids[1000:1010]
print(data_ids)

['9200249.zip', '2064129.zip', '15416.zip', '574.zip', '92030.zip', '618.zip', '571.zip', '9200211.zip', '254.zip', '2048605.zip']


In [10]:
# Example processing: save the first element to a UNZIP_DIR as an xml file
if not os.path.exists(UNZIP_DIR):
    os.makedirs(UNZIP_DIR)

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

In [5]:
def download_file(ftp_host, ftp_path, filename):
    zip_data = io.BytesIO()

    with ftplib.FTP(ftp_host) as ftp:
        ftp.login()  # Login as anonymous
        ftp.cwd(ftp_path)

        ftp.retrbinary(f'RETR {filename}', zip_data.write)
    
    zip_data.seek(0)
    return zip_data

# Function to unzip a file
def unzip_file(zip_data):
    extracted_files = []  # List to store file content

    with zipfile.ZipFile(zip_data, 'r') as zip_ref:
        for file_info in zip_ref.infolist():
            with zip_ref.open(file_info) as file:
                file_content = file.read()
                extracted_files.append(file_content)

    return extracted_files

def find_tier_information(tree, namespaces):
    # Initialize variables to store tier information
    content_tier = None

    # Find all hasBody elements
    has_body_elements = tree.findall('.//oa:hasBody', namespaces)

    for element in has_body_elements:
        resource = element.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource', '')
        
        # Check for content tier
        if 'contentTier' in resource:
            content_tier = resource.split('contentTier')[-1]

    return content_tier

# Function to download and process a ZIP file
def download_zip(filename):

    try:
        print(f"Starting download and processing for {filename}...")
        
        # Download the ZIP file into memory
        zip_data = download_file(FTP_HOST, FTP_PATH, filename)

        # Unzip and process the file
        extracted_files = unzip_file(zip_data)
        print(f"deleting zip file {filename}")
        del zip_data 

        ## TODO: Clicked docs 

        ## TODO: Sampled docs

        # print the filenames from the extracted files
        print(f"Extracted files: {(extracted_files[0])}")

        # make a subdirectory for the dataset
        dataset_dir = os.path.join(UNZIP_DIR, filename)
        dataset_dir = dataset_dir[:-4]
        if not os.path.exists(dataset_dir):
            os.makedirs(dataset_dir)
        
        # save the extracted files in the subdirectory
        for i, file_content in enumerate(extracted_files):
            with open(os.path.join(dataset_dir, f"{i}.xml"), 'wb') as file:
                file.write(file_content)

        print(f"Finished processing {filename}")
        del extracted_files

    except Exception as e:
        print(f"Error processing {filename}: {e}")

In [6]:
# Use ThreadPoolExecutor to download and process files in parallel
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(download_zip, filename) for filename in data_ids]

    # Use tqdm to show progress as futures are completed
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing ZIP files"):
        # This will raise exceptions if any occurred during processing
        future.result()

Starting download and processing for 9200249.zip...
Starting download and processing for 2064129.zip...
Starting download and processing for 15416.zip...
Starting download and processing for 574.zip...
Starting download and processing for 92030.zip...
Starting download and processing for 618.zip...
Starting download and processing for 571.zip...
Starting download and processing for 9200211.zip...
Starting download and processing for 254.zip...
Starting download and processing for 2048605.zip...


Processing ZIP files:   0%|          | 0/10 [00:00<?, ?it/s]

deleting zip file 15416.zip
Extracted files: b'<?xml version="1.0" encoding="UTF-8" standalone="yes"?><rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:dcterms="http://purl.org/dc/terms/" xmlns:edm="http://www.europeana.eu/schemas/edm/" xmlns:owl="http://www.w3.org/2002/07/owl#" xmlns:wgs84_pos="http://www.w3.org/2003/01/geo/wgs84_pos#" xmlns:skos="http://www.w3.org/2004/02/skos/core#" xmlns:rdaGr2="http://rdvocab.info/ElementsGr2/" xmlns:foaf="http://xmlns.com/foaf/0.1/" xmlns:ebucore="http://www.ebu.ch/metadata/ontologies/ebucore/ebucore#" xmlns:doap="http://usefulinc.com/ns/doap#" xmlns:odrl="http://www.w3.org/ns/odrl/2/" xmlns:cc="http://creativecommons.org/ns#" xmlns:ore="http://www.openarchives.org/ore/terms/" xmlns:svcs="http://rdfs.org/sioc/services#" xmlns:oa="http://www.w3.org/ns/oa#" xmlns:dqv="http://www.w3.org/ns/dqv#"><edm:ProvidedCHO rdf:about="http://data.europeana.eu/item/15416/Data_Library3_Library3_Batc

Processing ZIP files:  50%|█████     | 5/10 [00:02<00:01,  2.73it/s]

Finished processing 254.zip
Finished processing 15416.zip
Finished processing 9200249.zip
Finished processing 618.zip
Finished processing 574.zip


Processing ZIP files: 100%|██████████| 10/10 [00:02<00:00,  3.89it/s]

Finished processing 92030.zip
Finished processing 571.zip
Finished processing 9200211.zip
Finished processing 2048605.zip
Finished processing 2064129.zip


In [11]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Function to process a single subdirectory
def process_single_subdirectory(subdirectory_path):
    try:
        subdirectory = os.path.basename(subdirectory_path)
        print(f"Processing {subdirectory}...")

        # Get the size of the subdirectory
        size = len(os.listdir(subdirectory_path))
        print(f"Size of {subdirectory}: {size}")
        print(subdirectory_path)

        # Process RDF files in the subdirectory
        output = fe.parse_rdf_files(subdirectory_path, subdirectory)

        # Write the output to an XML file
        output_file = f'/home/sbasir/Thesis/Thesis/EDP/collected_data/{subdirectory}'
        fe.write_data(output, output_file)

        print(f"Finished processing {subdirectory}. Output written to {output_file}")

    except Exception as e:
        print(f"Error processing {subdirectory}: {e}")

# Threaded function to process all subdirectories in the given directory
def process_zip_threaded(directory):
    subdirectories = [os.path.join(directory, subdir) for subdir in os.listdir(directory) if os.path.isdir(os.path.join(directory, subdir))]

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(process_single_subdirectory, subdir): subdir for subdir in subdirectories}

        # Use tqdm to show progress as futures are completed
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing subdirectories"):
            try:
                future.result()
            except Exception as e:
                subdirectory = futures[future]
                print(f"Error processing subdirectory {subdirectory}: {e}")

# Example usage
process_zip_threaded('/home/sbasir/Thesis/Thesis/EDP/selected_data')

Processing 2048605...Processing 574...

Processing 618...
Processing 9200211...
Processing 2064129...
Processing 92030...
Processing 571...
Processing 15416...
Processing 254...
Processing 9200249...


Processing subdirectories:   0%|          | 0/10 [00:00<?, ?it/s]

Size of 15416: 1568
/home/sbasir/Thesis/Thesis/EDP/selected_data/15416
Size of 92030: 1560
/home/sbasir/Thesis/Thesis/EDP/selected_data/92030
Size of 2048605: 1547
/home/sbasir/Thesis/Thesis/EDP/selected_data/2048605
Size of 254: 1548
/home/sbasir/Thesis/Thesis/EDP/selected_data/254
Size of 9200211: 1554
/home/sbasir/Thesis/Thesis/EDP/selected_data/9200211
Size of 571: 1554
/home/sbasir/Thesis/Thesis/EDP/selected_data/571
Size of 574: 1567
/home/sbasir/Thesis/Thesis/EDP/selected_data/574
Size of 9200249: 1574
/home/sbasir/Thesis/Thesis/EDP/selected_data/9200249
Size of 618: 1558
/home/sbasir/Thesis/Thesis/EDP/selected_data/618
Size of 2064129: 1573
/home/sbasir/Thesis/Thesis/EDP/selected_data/2064129
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/9200211/1.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/9200211/2.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/9200211/3.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/

Processing subdirectories:  10%|█         | 1/10 [00:57<08:40, 57.89s/it]

Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/92030/1104.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/571/831.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/9200211/1371.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/571/832.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/92030/1105.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/618/1225.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/254/829.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2048605/679.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/571/833.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/9200211/1372.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/254/830.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/92030/1106.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/618/1226.xml
Data written 

...

Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/15416/282.xmlData written to /home/sbasir/Thesis/Thesis/EDP/collected_data/254/1323.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2048605/1183.xml

Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/254/1324.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/9200249/358.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/9200249/359.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2048605/1184.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/571/1316.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/15416/283.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/9200249/360.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/15416/284.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/571/1317.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/254/1325.xml
Data 

Processing subdirectories:  70%|███████   | 7/10 [00:58<00:11,  3.89s/it]

Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/574/1108.xmlData written to /home/sbasir/Thesis/Thesis/EDP/collected_data/9200249/1563.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/15416/1486.xml

Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/9200249/1564.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/15416/1487.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/9200249/1565.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/574/1109.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/15416/1488.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/15416/1489.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/9200249/1566.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/574/1110.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/15416/1490.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/9200249/1567

Processing subdirectories: 100%|██████████| 10/10 [00:58<00:00,  5.88s/it]

Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2064129/984.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2064129/985.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2064129/986.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2064129/987.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2064129/988.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2064129/989.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2064129/990.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2064129/991.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2064129/992.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2064129/993.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2064129/994.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2064129/995.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/20

In [8]:
# def process_zip(directory):
#     # for each subdirectory in the UNZIP_DIR print the size of the subdirectory
#     for subdirectory in os.listdir(directory):
#         subdirectory_path = os.path.join(directory, subdirectory)
#         print(f"Size of {subdirectory}: {len(os.listdir(subdirectory_path))}")
#         print(subdirectory_path)
#         output = fe.parse_rdf_files(subdirectory_path)
#         fe.write_data(output, f'/home/sbasir/Thesis/Thesis/EDP/collected_data/{subdirectory}.xml')

In [9]:
# process_zip('/home/sbasir/Thesis/Thesis/EDP/selected_data')

Size of 2048605: 1547
/home/sbasir/Thesis/Thesis/EDP/selected_data/2048605


Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2048605.xml/1.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2048605.xml/2.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2048605.xml/3.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2048605.xml/4.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2048605.xml/5.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2048605.xml/6.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2048605.xml/7.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2048605.xml/8.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2048605.xml/9.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2048605.xml/10.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2048605.xml/11.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2048605.xml/12.xml
Data written to /home/sbasir/Thesis/T

KeyboardInterrupt: 

In [ ]:
# # Usage
# directory = '/home/sbasir/Thesis/Thesis/EDP/selected_data/254'  # Change this to your directory containing RDF/XML files
# solr_xml_data = fe.parse_rdf_files(directory)
# # solr_xml_data = fe.remove_duplicates(solr_xml_data)
# output_file = '/home/sbasir/Thesis/Thesis/EDP/testing.xml'
# fe.write_data(solr_xml_data, output_file)